# Phase 3, experiment 12 (part 2): blending the averaged GNN with LightGBM

This notebook is the same as `14_gnn_blend.ipynb`, but uses the
average of 5 GNNs instead of a single one.

`blend = (1 - w) x LightGBM rank + w x GNN rank`, where `w` is the GNN's weight.

**Results:** In notebook 14 the folds and the public leaderboard disagreed: the
leaderboard rose with the GNN weight (0.94660, 0.95037, 0.95325 at weights 0, 0.1, 0.2) while the fold
mean fell slightly. So the fold table here shows how the blend behaves on the validation periods, but
the leaderboard is the only evidence on what the GNN adds on the test period.

## 1. Load

In [9]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

SEED = 42
DATA_DIR = "../data"
OUT_DIR = "../outputs"

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
raw_cols = [c for c in train.columns if c.startswith("feat_")]

labeled = train[train["label"].notna()].copy()
labeled["label"] = labeled["label"].astype(int)

gnn_fold = pd.read_csv(f"{OUT_DIR}/gnn5_fold_predictions.csv")
gnn_test = pd.read_csv(f"{OUT_DIR}/gnn5_test_predictions.csv")

## 2. Blend on the folds

In [10]:
FOLDS = [
    {"name": "fold 1", "train_last": 14, "valid_last": 21},
    {"name": "fold 2", "train_last": 21, "valid_last": 28},
    {"name": "fold 3", "train_last": 28, "valid_last": 35},
]
fold_names = [f["name"] for f in FOLDS]
GNN_WEIGHTS = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 1.0]

def to_rank(scores):
    # Position of each score (1 = lowest) divided by the count, so it runs from 0 to 1
    return pd.Series(scores).rank().values / len(scores)

score_rows = []
for fold in FOLDS:
    tr = labeled[labeled["time_step"] <= fold["train_last"]]
    va = labeled[(labeled["time_step"] > fold["train_last"]) & (labeled["time_step"] <= fold["valid_last"])]

    model = lgb.LGBMClassifier(random_state=SEED, verbose=-1)
    model.fit(tr[raw_cols], tr["label"])
    lgbm_rank = to_rank(model.predict_proba(va[raw_cols])[:, 1])

    this_fold = gnn_fold[gnn_fold["fold"] == fold["name"]].set_index("txId")["gnn_score"]
    gnn_rank = to_rank(this_fold.loc[va["txId"]].values)

    in_24 = (va["time_step"] == 24).values
    for w in GNN_WEIGHTS:
        blend = (1 - w) * lgbm_rank + w * gnn_rank
        row = {"gnn_weight": w, "fold": fold["name"], "val_auc": roc_auc_score(va["label"], blend)}
        if fold["name"] == "fold 2":
            row["auc_step_24"] = roc_auc_score(va["label"][in_24], blend[in_24])
        score_rows.append(row)

scores = pd.DataFrame(score_rows)
results = scores.pivot(index="gnn_weight", columns="fold", values="val_auc")
results["mean"] = results[fold_names].mean(axis=1)
results["change_mean"] = results["mean"] - results.loc[0.0, "mean"]
results["change_fold2"] = results["fold 2"] - results.loc[0.0, "fold 2"]
results["auc_step_24"] = scores.dropna(subset=["auc_step_24"]).set_index("gnn_weight")["auc_step_24"]
results.round(4)

fold,fold 1,fold 2,fold 3,mean,change_mean,change_fold2,auc_step_24
gnn_weight,,,,,,,
0.0,0.9888,0.9572,0.9965,0.9808,0.0000,0.0000,0.7866
0.1,0.9900,0.9548,0.9978,0.9809,0.0001,-0.0024,0.7806
0.2,0.9906,0.9517,0.9983,0.9802,-0.0006,-0.0055,0.7740
0.3,0.9903,0.9479,0.9983,0.9788,-0.0020,-0.0093,0.7637
0.4,0.9892,0.9437,0.9980,0.9770,-0.0039,-0.0135,0.7496
0.5,0.9872,0.9392,0.9974,0.9746,-0.0062,-0.0180,0.7365
0.6,0.9844,0.9345,0.9966,0.9718,-0.0090,-0.0227,0.7234
0.7,0.9805,0.9299,0.9957,0.9687,-0.0121,-0.0273,0.7085
1.0,0.9615,0.9122,0.9857,0.9531,-0.0277,-0.0450,0.6664


## 3. Submissions

LightGBM is retrained on steps 1 to 35 once, and one submission file is saved for each weight listed
in `SUBMIT_WEIGHTS`

In [ ]:
# GNN weight -> submission number (numbers must match the experiment log).
SUBMIT_WEIGHTS = {0.6: 12, 0.7: 13}   # final choice is 0.4 (sub10)

final_model = lgb.LGBMClassifier(random_state=SEED, verbose=-1)
final_model.fit(labeled[raw_cols], labeled["label"])
lgbm_test_rank = to_rank(final_model.predict_proba(test[raw_cols])[:, 1])
gnn_test_rank = to_rank(gnn_test.set_index("txId")["gnn_score"].loc[test["txId"]].values)

for w in SUBMIT_WEIGHTS:
    test_pred = (1 - w) * lgbm_test_rank + w * gnn_test_rank
    submission = pd.DataFrame({"index": test["index"], "target": test_pred})

    assert len(submission) == 15329, "wrong number of rows"
    assert list(submission.columns) == ["index", "target"], "wrong columns"
    assert submission["target"].notna().all(), "NaN in predictions"
    assert (submission["index"].values == sample_sub["index"].values).all(), "index does not match sample submission"

    number = SUBMIT_WEIGHTS[w]   # the submission number chosen above
    path = f"../submissions/sub{number:02d}_gnn5_blend_w{int(round(w * 10)):02d}.csv"
    submission.to_csv(path, index=False)
    print("Saved", path)

Saved ../submissions/sub12_gnn5_blend_w06.csv
Saved ../submissions/sub13_gnn5_blend_w07.csv


## Findings

- **How does the averaged-GNN blend behave on the folds, compared with the single-GNN blend (notebook 14)?**
  Slightly better at every weight both notebooks tried: fold mean 0.9809 against 0.9808 at weight 0.1,
  0.9802 against 0.9799 at 0.2, and 0.9746 against 0.9734 at 0.5. The pattern is the same as before:
  weight 0.1 is level with LightGBM alone (mean +0.0001, fold 2 -0.0024), and higher weights raise fold 3
  slightly (up to +0.0018) but lower fold 2 more (-0.0055 at 0.2, -0.0093 at 0.3), so the mean falls.
  By the rule used so far (raise the mean without lowering fold 2), no weight passes.
- **What happens on step 24?** It falls steadily as the GNN weight rises: 0.787 for LightGBM alone,
  0.774 at 0.2, 0.764 at 0.3 and 0.666 for the averaged GNN alone (0.653 for the single GNN).
- **Public leaderboard AUC for the submitted files:** `sub08_gnn5_blend_w02.csv` (weight 0.2) scored
  0.95369, 0.0004 above the single-GNN blend at the same weight (`sub07`, 0.95325). `sub09_gnn5_blend_w03.csv`
  (weight 0.3) scored 0.95581, the best so far. Across all blend submissions the public score has risen
  with every increase in GNN weight: 0.94660 (0), 0.95037 (0.1), 0.95325 / 0.95369 (0.2), 0.95581 (0.3),
  while the fold mean has fallen slightly (0.9808 to 0.9788).
- **Later submissions:** weight 0.4 (`sub10_gnn5_blend_w04.csv`) scored 0.95654 and weight 0.5
  (`sub11_gnn5_blend_w05.csv`) 0.95604, so the public score peaks around 0.4.
- **Decision: which GNN (single or averaged) and which weight for the final model?** The average of 5
  GNNs, blended with LightGBM at GNN weight 0.4 (`sub10`): it has the best public score, at the top of the
  curve rather than on its slope. The cost is a lower fold 2 score (0.9437 against 0.9572 for LightGBM
  alone), so the fold results warn that it could do worse if the private test rows contain more new
  kinds of illicit behaviour.
